<a href="https://colab.research.google.com/github/jhajagos/SupportingConceptSetGeneration/blob/main/UMLS_Vocabulary_Nested_Structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Build concept vocabulary files to be imported for reasoning

### Setup environment

In [1]:
import pandas as pd
import pyspark
import json
import logging

In [2]:
UMLS_DIRECTORY = "/content/drive/MyDrive/umls/2025AA/"
UMLS_OUTPUT_DIRECTORY = "/content/drive/MyDrive/umls/2025AA/export/"

In [3]:
OHSDI_VOCABULARY_PATH = "/content/drive/MyDrive/OHDSI/vocabulary/20250317/export/"

In [4]:
spark = pyspark.sql.SparkSession.builder\
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
spark = pyspark.sql.SparkSession.builder\
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

In [7]:
def attach_catalog_dict(spark_ptr, table_catalog, domains_to_exclude=None):
    """Takes an output dictionary from the OHDSI mapper and maps tables in the domain"""

    db_cat = {}
    if domains_to_exclude is not None:
        filtered_domains = [dn for dn in table_catalog if dn not in domains_to_exclude]
    else:
        filtered_domains = list(table_catalog.keys())

    for domain in filtered_domains:
        for table in table_catalog[domain]:
            location = table_catalog[domain][table]
            if domain not in db_cat:
                db_cat[domain] = {}
            logging.info(f"Loading '{location}' and attaching to namespace as '{table}'")
            db_cat[domain][table] = spark_ptr.read.parquet(location)
            db_cat[domain][table].createOrReplaceTempView(table)

    return db_cat

### Build nested structure that emulates UMLS API response in bulk

#### Build Spark Dataframes

In [8]:
with open(UMLS_OUTPUT_DIRECTORY + "umls_generated_tables.json", "r") as f:
  catalog_dict = json.load(f)

ct = attach_catalog_dict(spark, catalog_dict)
ct

{'umls': {'MRCONSO': DataFrame[CUI: string, LAT: string, TS: string, LUI: string, STT: string, SUI: string, ISPREF: string, AUI: string, SAUI: bigint, SCUI: string, SDUI: string, SAB: string, TTY: string, CODE: string, STR: string, SRL: int, SUPPRESS: string, CVF: int, dummy: string],
  'MRHIER': DataFrame[CUI: string, AUI: string, CXN: int, PAUI: string, SAB: string, RELA: string, PTR: string, HCD: string, CVF: string, dummy: string],
  'MRDEF': DataFrame[CUI: string, AUI: string, ATUI: string, SATUI: bigint, SAB: string, DEF: string, SUPPRESS: string, CVF: string, dummy: string],
  'MRSAB': DataFrame[VCUI: string, RCUI: string, VSAB: string, RSAB: string, SON: string, SF: string, SVER: string, VSTART: string, VEND: string, IMETA: string, RMETA: string, SLC: string, SCC: string, SRL: int, TFR: int, CFR: int, CXTY: string, TTYL: string, ATNL: string, LAT: string, CENC: string, CURVER: string, SABIN: string, SSN: string, SCIT: string, dummy: string],
  'MRSAT': DataFrame[CUI: string, LU

In [9]:
# This is an example build that show the UMLS structure
"""
{'code': 'HP:0033630',
 'name': 'Brain fog',
 'vocabulary': 'HPO',
 'concepts': {'C0015676': {'concept_uri': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0015676',
   'semantic_type': 'Mental or Behavioral Dysfunction',
   'definitions': {'MSH': 'A condition of low alertness or cognitive impairment, usually associated with prolonged mental activities or stress.',
    'HPO': 'Brain fog is a type of transient cognitive dysfunction that comprises a constellation of symptoms that impair intellectual functioning to a level that interferes with daily activities, commonly including forgetfulness, mental slowness, difficulty thinking or focusing, a perceived slowing of mental processing speed, inability to find the right words, a sensation that the mind went blank or is cloudy". Brain fog tends to recur and may be triggered by factors such as physical fatigue, lack of sleep, and prolonged standing or may appear to occur spontaneously." [PMID:23999934, PMID:30452327, PMID:32857796, PMID:32984564, PMID:33199820]'}}},
 'attributes': {'HPO_COMMENT': 'Brain fog has been reported in a range of diseases including myalgic encephalomyelitis/chronic fatigue syndrome, non-celiac gluten disease, postural tachycardia syndrome, lupus, and COVID-19.'},
 'relationships': [{'relationship_source': 'HPO',
   'relationship_type': 'SY',
   'relationship_type_description': '',
   'relationship_target': 'Brain fog',
   'relationship_target_code': 'A33194066'},
  {'relationship_source': 'HPO',
   'relationship_type': 'SY',
   'relationship_type_description': '',
   'relationship_target': 'Mental clouding',
   'relationship_target_code': 'A33193023'},
  {'relationship_source': 'HPO',
   'relationship_type': 'SY',
   'relationship_type_description': '',
   'relationship_target': 'Mental fog',
   'relationship_target_code': 'A33194547'},
  {'relationship_source': 'HPO',
   'relationship_type': 'SY',
   'relationship_type_description': '',
   'relationship_target': 'Mental fatigue',
   'relationship_target_code': 'A33194067'},
  {'relationship_source': 'HPO',
   'relationship_type': 'CHD',
   'relationship_type_description': 'isa',
   'relationship_target': 'Cognitive impairment',
   'relationship_target_code': 'HP:0100543'},
  {'relationship_source': 'HPO',
   'relationship_type': 'SY',
   'relationship_type_description': '',
   'relationship_target': 'Brain fog',
   'relationship_target_code': 'A33194066'},
  {'relationship_source': 'HPO',
   'relationship_type': 'SY',
   'relationship_type_description': '',
   'relationship_target': 'Brain fog',
   'relationship_target_code': 'A33194066'}],
 'parents': [{'id': 'HP:0100543', 'name': 'Cognitive impairment'}],
 'children': [],
 'ancesotors': [{'id': 'V-HPO', 'name': 'Human Phenotype Ontology'},
  {'id': 'HP:0100543', 'name': 'Cognitive impairment'},
  {'id': 'HP:0000118', 'name': 'Phenotypic abnormality'},
  {'id': 'HP:0000707', 'name': 'Abnormality of the nervous system'},
  {'id': 'HP:0000001', 'name': 'All'},
  {'id': 'HP:0012638', 'name': 'Abnormal nervous system physiology'},
  {'id': 'HP:0011446', 'name': 'Abnormality of mental function'}],
 'descendants': []}
"""

'\n{\'code\': \'HP:0033630\',\n \'name\': \'Brain fog\',\n \'vocabulary\': \'HPO\',\n \'concepts\': {\'C0015676\': {\'concept_uri\': \'https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0015676\',\n   \'semantic_type\': \'Mental or Behavioral Dysfunction\',\n   \'definitions\': {\'MSH\': \'A condition of low alertness or cognitive impairment, usually associated with prolonged mental activities or stress.\',\n    \'HPO\': \'Brain fog is a type of transient cognitive dysfunction that comprises a constellation of symptoms that impair intellectual functioning to a level that interferes with daily activities, commonly including forgetfulness, mental slowness, difficulty thinking or focusing, a perceived slowing of mental processing speed, inability to find the right words, a sensation that the mind went blank or is cloudy". Brain fog tends to recur and may be triggered by factors such as physical fatigue, lack of sleep, and prolonged standing or may appear to occur spontaneously." [PMID:23

In [32]:
sab = "ICD10CM" # UMLS Name
ohdsi_vocabulary = "ICD10CM" # OHDSI Name
tty = ("PT", "HT")

subset_vocab_sdf = spark.sql(f"select * from MRCONSO where TTY in {tty} and SAB='{sab}' order by CODE")
subset_vocab_sdf.createOrReplaceTempView("subset_vocab")

print(f"Total rows: {subset_vocab_sdf.count()}")
subset_vocab_sdf.limit(10).toPandas()

Total rows: 97903


,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,SDUI,SAB,TTY,CODE,STR,SRL,SUPPRESS,CVF,dummy
0,C0008354,ENG,P,L0008354,PF,S0024863,N,A17850096,NaN,None,A00,ICD10CM,HT,A00,Cholera,4,N,256.0,None
1,C0178238,ENG,S,L9592854,PF,S12054926,Y,A18910968,NaN,None,A00-A09,ICD10CM,HT,A00-A09,Intestinal infectious diseases (A00-A09),4,N,NaN,None
2,C0694449,ENG,P,L9596054,PF,S12053689,Y,A18905737,NaN,None,A00-B99,ICD10CM,HT,A00-B99,Certain infectious and parasitic diseases (A00...,4,N,NaN,None
3,C0494021,ENG,P,L0616536,PF,S0837144,N,A17862685,NaN,None,A00.0,ICD10CM,PT,A00.0,"Cholera due to Vibrio cholerae 01, biovar chol...",4,N,NaN,None
4,C0343372,ENG,S,L0616537,PF,S0837145,N,A17786165,NaN,None,A00.1,ICD10CM,PT,A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor",4,N,NaN,None
5,C0008354,ENG,S,L0543280,PF,S0617259,N,A17862687,NaN,None,A00.9,ICD10CM,PT,A00.9,"Cholera, unspecified",4,N,256.0,None
6,C0275976,ENG,P,L0179317,VC,S0243831,Y,A17798886,NaN,None,A01,ICD10CM,HT,A01,Typhoid and paratyphoid fevers,4,N,256.0,None
7,C0041466,ENG,P,L0041468,VC,S0000262,N,A17862688,NaN,None,A01.0,ICD10CM,HT,A01.0,Typhoid fever,4,N,256.0,None
8,C0041466,ENG,S,L9375097,PF,S11687058,Y,A17773402,NaN,None,A01.00,ICD10CM,PT,A01.00,"Typhoid fever, unspecified",4,N,256.0,None
9,C2880086,ENG,P,L9385296,PF,S11687059,Y,A17798887,NaN,None,A01.01,ICD10CM,PT,A01.01,Typhoid meningitis,4,N,256.0,None


In [11]:
subset_def_sdf = spark.sql("""select distinct sv.CUI, md.DEF, md.SAB from subset_vocab sv
join MRDEF md on sv.CUI = md.CUI
join (select distinct SAB from MRCONSO where LAT='ENG') eng on md.SAB = eng.SAB order by CUI, md.SAB
""")
subset_def_sdf.createOrReplaceTempView("subset_def")

print(f"Total rows: {subset_def_sdf.count()}")

subset_def_sdf.limit(10).toPandas()

Total rows: 10949


,CUI,DEF,SAB
0,C0000727,A sudden onset of abdominal pain with associat...,HPO
1,C0000727,A clinical syndrome with acute abdominal pain ...,MSH
2,C0000737,An unpleasant sensation characterized by physi...,HPO
3,C0000737,<p>Your abdomen extends from below your chest ...,MEDLINEPLUS
4,C0000737,"Sensation of discomfort, distress, or agony in...",MSH
5,C0000737,Painful sensation in the abdominal region.,NCI
6,C0000768,Structural or functional abnormalities of the ...,HPO
7,C0000768,Malformations of organs or body parts during d...,MSH
8,C0000768,"Any abnormality, anatomical or biochemical, ev...",NCI
9,C0000786,the natural premature expulsion from the uteru...,CSP


In [12]:
subset_sty_sdf = spark.sql("""
  select distinct ms.CUI, ms.STN, ms.STY from subset_vocab sv
  join MRSTY ms on sv.CUI = ms.CUI order by CUI, ms.STN
  """)
subset_sty_sdf.createOrReplaceTempView("subset_sty")

print(f"Total rows: {subset_sty_sdf.count()}")

subset_sty_sdf.limit(10).toPandas()


Total rows: 95337


,CUI,STN,STY
0,C0000727,A2.2.2,Sign or Symptom
1,C0000737,A2.2.2,Sign or Symptom
2,C0000768,A1.2.2.1,Congenital Abnormality
3,C0000770,A1.2.2,Anatomical Abnormality
4,C0000786,B2.2.1.2,Pathologic Function
5,C0000809,B2.2.1.2,Pathologic Function
6,C0000814,B2.2.1.2.1,Disease or Syndrome
7,C0000821,B2.2.1.2,Pathologic Function
8,C0000832,B2.2.1.2,Pathologic Function
9,C0000889,B2.2.1.2.1,Disease or Syndrome


In [13]:
subset_rel_sdf = spark.sql("""
select mr.AUI1, mr.REL, mr.RELA, sv2.AUI, sv2.TTY, sv2.STR, sv2.CODE from MRREL mr
  join subset_vocab sv1 on mr.AUI1 = sv1.AUI
  join subset_vocab sv2 on mr.AUI2 = sv2.AUI
  order by mr.AUI1, mr.RELA, mr.SAB, mr.RELA
""")
subset_rel_sdf.createOrReplaceTempView("subset_rel")

print(f"Total rows: {subset_rel_sdf.count()}")

subset_rel_sdf.limit(10).toPandas()

Total rows: 195804


,AUI1,REL,RELA,AUI,TTY,STR,CODE
0,A17773402,PAR,None,A17862688,HT,Typhoid fever,A01.0
1,A17773404,PAR,None,A17862688,HT,Typhoid fever,A01.0
2,A17773405,CHD,None,A17798891,PT,Salmonella arthritis,A02.23
3,A17773405,CHD,None,A17824692,PT,Salmonella pyelonephritis,A02.25
4,A17773405,CHD,None,A17786167,PT,Salmonella pneumonia,A02.22
5,A17773405,CHD,None,A17798892,PT,Salmonella osteomyelitis,A02.24
6,A17773405,CHD,None,A17811769,PT,Salmonella meningitis,A02.21
7,A17773405,CHD,None,A17824693,PT,Salmonella with other localized infection,A02.29
8,A17773405,CHD,None,A17811768,PT,"Localized salmonella infection, unspecified",A02.20
9,A17773405,PAR,None,A17837351,HT,Other salmonella infections,A02


In [14]:
from pyspark.sql import functions as F

In [15]:
mrconso_sdf = ct["umls"]["MRCONSO"]
mrhier_sdf = ct["umls"]["MRHIER"]

In [16]:
mrconso_sdf = ct["umls"]["MRCONSO"]
mrhier_sdf = ct["umls"]["MRHIER"]

In [17]:
subset_mrhier_sdf = mrhier_sdf.alias("mh").join(subset_vocab_sdf.alias("sv"), F.col("mh.AUI") == F.col("sv.AUI")).select("mh.*")
subset_mrhier_sdf.createOrReplaceTempView("subset_mrhier")

print(f"Total rows: {subset_mrhier_sdf.count()}")

subset_mrhier_sdf.limit(10).toPandas()

Total rows: 97903


,CUI,AUI,CXN,PAUI,SAB,RELA,PTR,HCD,CVF,dummy
0,C4267902,A27155001,1,A27153115,ICD10CM,None,A20098492.A18921516.A18916268.A17850773.A17838...,None,None,None
1,C4268487,A27152818,1,A17813523,ICD10CM,None,A20098492.A18916316.A18916317.A17775120.A17864...,None,None,None
2,C4268511,A27155043,1,A27153529,ICD10CM,None,A20098492.A18916316.A18916317.A17813539.A17775...,None,None,None
3,C4268644,A27154302,1,A27150129,ICD10CM,None,A20098492.A20161433.A18919038.A17865000.A27150129,None,None,None
4,C4268677,A27153933,1,A27154308,ICD10CM,None,A20098492.A18905900.A18908469.A17865120.A27154308,None,None,None
5,C4268931,A27152851,1,A27149771,ICD10CM,None,A20098492.A18908579.A18916487.A17789705.A27149771,None,None,None
6,C4269377,A27154767,1,A27152492,ICD10CM,None,A20098492.A18906120.A18911375.A17803504.A17790...,None,None,None
7,C4269587,A27151349,1,A27151724,ICD10CM,None,A20098492.A18906120.A18911375.A17816409.A17777...,None,None,None
8,C4269665,A27153254,1,A27151736,ICD10CM,None,A20098492.A18906120.A18906744.A17834007.A27151...,None,None,None
9,C4269726,A27152519,1,A27151014,ICD10CM,None,A20098492.A18906120.A18906744.A17834007.A27151...,None,None,None


In [18]:
subset_mrhier_sdf = subset_mrhier_sdf.withColumn("ptr_list", F.split(F.col("PTR"), r"\."))
subset_mrhier_sdf.limit(10).toPandas()

,CUI,AUI,CXN,PAUI,SAB,RELA,PTR,HCD,CVF,dummy,ptr_list
0,C4267902,A27155001,1,A27153115,ICD10CM,None,A20098492.A18921516.A18916268.A17850773.A17838...,None,None,None,"[A20098492, A18921516, A18916268, A17850773, A..."
1,C4268487,A27152818,1,A17813523,ICD10CM,None,A20098492.A18916316.A18916317.A17775120.A17864...,None,None,None,"[A20098492, A18916316, A18916317, A17775120, A..."
2,C4268511,A27155043,1,A27153529,ICD10CM,None,A20098492.A18916316.A18916317.A17813539.A17775...,None,None,None,"[A20098492, A18916316, A18916317, A17813539, A..."
3,C4268644,A27154302,1,A27150129,ICD10CM,None,A20098492.A20161433.A18919038.A17865000.A27150129,None,None,None,"[A20098492, A20161433, A18919038, A17865000, A..."
4,C4268677,A27153933,1,A27154308,ICD10CM,None,A20098492.A18905900.A18908469.A17865120.A27154308,None,None,None,"[A20098492, A18905900, A18908469, A17865120, A..."
5,C4268931,A27152851,1,A27149771,ICD10CM,None,A20098492.A18908579.A18916487.A17789705.A27149771,None,None,None,"[A20098492, A18908579, A18916487, A17789705, A..."
6,C4269377,A27154767,1,A27152492,ICD10CM,None,A20098492.A18906120.A18911375.A17803504.A17790...,None,None,None,"[A20098492, A18906120, A18911375, A17803504, A..."
7,C4269587,A27151349,1,A27151724,ICD10CM,None,A20098492.A18906120.A18911375.A17816409.A17777...,None,None,None,"[A20098492, A18906120, A18911375, A17816409, A..."
8,C4269665,A27153254,1,A27151736,ICD10CM,None,A20098492.A18906120.A18906744.A17834007.A27151...,None,None,None,"[A20098492, A18906120, A18906744, A17834007, A..."
9,C4269726,A27152519,1,A27151014,ICD10CM,None,A20098492.A18906120.A18906744.A17834007.A27151...,None,None,None,"[A20098492, A18906120, A18906744, A17834007, A..."


In [19]:
subset_mrhier_exploded_sdf = subset_mrhier_sdf.alias("mh").select(F.col("AUI").alias("child_AUI"), F.col("RELA"),
                                  F.posexplode(F.col("ptr_list"))).filter(F.col("CXN")  == F.lit(1)).join(subset_vocab_sdf.alias("hpo").select("AUI","CUI","TTY", "CODE", "STR"),
                                                                        on=F.col("col")==F.col("hpo.AUI")).distinct().orderBy(["child_AUI", "RELA", "pos"], ascending=[True, True, False])
print(f"Total rows: {subset_mrhier_exploded_sdf.count()}")

subset_mrhier_exploded_sdf.limit(10).toPandas()

Total rows: 581993


,child_AUI,RELA,pos,col,AUI,CUI,TTY,CODE,STR
0,A17773402,None,4,A17862688,A17862688,C0041466,HT,A01.0,Typhoid fever
1,A17773402,None,3,A17798886,A17798886,C0275976,HT,A01,Typhoid and paratyphoid fevers
2,A17773402,None,2,A18910968,A18910968,C0178238,HT,A00-A09,Intestinal infectious diseases (A00-A09)
3,A17773402,None,1,A18905737,A18905737,C0694449,HT,A00-B99,Certain infectious and parasitic diseases (A00...
4,A17773402,None,0,A20098492,A20098492,C2880081,HT,ICD-10-CM,ICD-10-CM TABULAR LIST of DISEASES and INJURIES
5,A17773404,None,4,A17862688,A17862688,C0041466,HT,A01.0,Typhoid fever
6,A17773404,None,3,A17798886,A17798886,C0275976,HT,A01,Typhoid and paratyphoid fevers
7,A17773404,None,2,A18910968,A18910968,C0178238,HT,A00-A09,Intestinal infectious diseases (A00-A09)
8,A17773404,None,1,A18905737,A18905737,C0694449,HT,A00-B99,Certain infectious and parasitic diseases (A00...
9,A17773404,None,0,A20098492,A20098492,C2880081,HT,ICD-10-CM,ICD-10-CM TABULAR LIST of DISEASES and INJURIES


In [20]:
subset_mrhier_rela_stubs_sdf = subset_mrhier_exploded_sdf.select("RELA").distinct().orderBy("RELA")

subset_mrhier_rela_stubs_sdf.createOrReplaceTempView("subset_mrhier_rela_stubs")

print(f"Total rows: {subset_mrhier_rela_stubs_sdf.count()}")

subset_mrhier_rela_stubs_sdf.limit(10).toPandas()

Total rows: 1


,RELA
0,None


#### Convert Spark Dataframes into dictionaries

In [21]:
subset_vocab_dict = subset_vocab_sdf.select("AUI", "LUI", "CUI", "ISPREF", "TTY", "SAB", "CODE", "STR").toPandas().to_dict("records")
subset_vocab_dict[0:2]

[{'AUI': 'A17850096',
  'LUI': 'L0008354',
  'CUI': 'C0008354',
  'ISPREF': 'N',
  'TTY': 'HT',
  'SAB': 'ICD10CM',
  'CODE': 'A00',
  'STR': 'Cholera'},
 {'AUI': 'A18910968',
  'LUI': 'L9592854',
  'CUI': 'C0178238',
  'ISPREF': 'Y',
  'TTY': 'HT',
  'SAB': 'ICD10CM',
  'CODE': 'A00-A09',
  'STR': 'Intestinal infectious diseases (A00-A09)'}]

In [22]:
def key_nest_list_sdf(sdf, key_column):
  """Builds a dictionary that points to a nested list"""

  list_dict = sdf.toPandas().to_dict(orient="records")

  dict_nested_list = {}

  for row_dict in list_dict:
    key = row_dict[key_column]
    row_dict.pop(key_column)

    if key not in dict_nested_list:
      dict_nested_list[key] = [row_dict]
    else:
      dict_nested_list[key] += [row_dict]

  return dict_nested_list


In [23]:
cui_sty_nested_list_dict = key_nest_list_sdf(subset_sty_sdf, "CUI")
len(cui_sty_nested_list_dict)

95314

In [24]:
cui_def_nested_list_dict = key_nest_list_sdf(subset_def_sdf, "CUI")
len(cui_def_nested_list_dict)

6538

In [25]:
aui_rel_nested_list_dict = key_nest_list_sdf(subset_rel_sdf, "AUI1")
len(aui_rel_nested_list_dict)

97903

In [26]:
hier_rela = subset_mrhier_rela_stubs_sdf.toPandas().to_dict("records")
hier_rela


[{'RELA': None}]

In [27]:
hier_rela_nested_list_dict = {}
for rela_d in hier_rela:
  rela = rela_d["RELA"]

  if rela is None:
    hier_rela_nested_list_dict["None"] = key_nest_list_sdf(subset_mrhier_exploded_sdf.filter(F.col("RELA").isNull()), "child_AUI")
  else:
    hier_rela_nested_list_dict[rela] = key_nest_list_sdf(subset_mrhier_exploded_sdf.filter(F.col("RELA") == F.lit(rela)), "child_AUI")

for key in hier_rela_nested_list_dict:
  print(f"{key}: {len(hier_rela_nested_list_dict[key])}")

None: 97902


#### Nest dictionaries

In [28]:
for row_dict in subset_vocab_dict:

  if row_dict["CUI"] in cui_sty_nested_list_dict:
    row_dict["STY"] = cui_sty_nested_list_dict[row_dict["CUI"]]
  if row_dict["CUI"] in cui_def_nested_list_dict:
    row_dict["DEF"] = cui_def_nested_list_dict[row_dict["CUI"]]

  if row_dict["AUI"] in aui_rel_nested_list_dict:
    row_dict["REL"] = aui_rel_nested_list_dict[row_dict["AUI"]]

  row_dict["hierarchies"] = {}
  for key in hier_rela_nested_list_dict:
    if row_dict["AUI"] in hier_rela_nested_list_dict[key]:
      cleaned_rela = []
      for c in hier_rela_nested_list_dict[key][row_dict["AUI"]]:
        c.pop("RELA")
        cleaned_rela += [c]
      row_dict["hierarchies"][key] = cleaned_rela


subset_vocab_dict[0]

{'AUI': 'A17850096',
 'LUI': 'L0008354',
 'CUI': 'C0008354',
 'ISPREF': 'N',
 'TTY': 'HT',
 'SAB': 'ICD10CM',
 'CODE': 'A00',
 'STR': 'Cholera',
 'STY': [{'STN': 'B2.2.1.2.1', 'STY': 'Disease or Syndrome'}],
 'DEF': [{'DEF': 'acute diarrheal disease endemic in India and southeast Asia whose causative agent is Vibrio cholerae; can lead to severe dehydration in a matter of hours unless quickly treated.',
   'SAB': 'CSP'},
  {'DEF': '<p>Cholera is a bacterial infection that causes <a href="https://medlineplus.gov/diarrhea.html">diarrhea</a>. The cholera bacterium is usually found in water or food that has been contaminated by feces (poop). Cholera is rare in the US. You may get it if you travel to parts of the world with poor water and sewage treatment. Outbreaks can also happen after disasters. The disease is not likely to spread directly from person to person.</p> <p>Cholera infections are often mild. Some people don\'t have any symptoms. If you do get symptoms, they usually start 2 to 

### Add OHDSI Concept Linkage

#### Build Spark Dataframes

In [29]:
concept_sdf = spark.read.parquet(OHSDI_VOCABULARY_PATH + "/concept.parquet")
concept_sdf.createOrReplaceTempView("concept")

print(f"Total rows read: {concept_sdf.count()}")
concept_sdf.limit(10).toPandas()

Total rows read: 6929890


,concept_id,concept_name,domain_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason,vocabulary_id
0,44021505,Panoxyl,Drug,Brand Name,None,OMOP1016136,19700101,20180801,D,RxNorm Extension
1,44083482,Benzoyl Peroxide Topical Gel [Panoxyl],Drug,Branded Drug Form,None,OMOP1078113,20170824,20180801,D,RxNorm Extension
2,44107281,Spectinomycin Injectable Solution,Drug,Clinical Drug Form,None,OMOP1101912,20170824,20240125,U,RxNorm Extension
3,35411914,Minoxidil 0.05 MG/MG,Drug,Clinical Drug Comp,None,OMOP1145499,20170824,20240125,U,RxNorm Extension
4,40859601,Nitrofurantoin 10 MG/ML [Furadantin],Drug,Branded Drug Comp,None,OMOP2057563,20170824,20240125,U,RxNorm Extension
5,40922810,Diclofenac 100 MG [Diclo Recip],Drug,Branded Drug Comp,None,OMOP2120772,20170730,20170823,D,RxNorm Extension
6,40954063,Diclofenac 100 MG [Diclo],Drug,Branded Drug Comp,None,OMOP2152025,20170730,20170823,D,RxNorm Extension
7,41178578,Minoxidil 0.05 MG/MG Topical Gel,Drug,Clinical Drug,None,OMOP2376540,20170824,20240125,U,RxNorm Extension
8,40799073,Proquazone,Drug,Ingredient,None,OMOP2721409,20170718,20240125,U,RxNorm Extension
9,42620194,Dibasic potassium phosphate / potassium phosph...,Drug,Clinical Drug Form,None,OMOP2800532,20170824,20240125,U,RxNorm Extension


In [31]:
concept_relationship_sdf = spark.read.parquet(OHSDI_VOCABULARY_PATH + "/concept_relationship.parquet")
concept_relationship_sdf.createOrReplaceTempView("concept_relationship")

print(f"Total rows read: {concept_relationship_sdf.count()}")
concept_relationship_sdf.limit(10).toPandas()

Total rows read: 44453650


,concept_id_1,concept_id_2,valid_start_date,valid_end_date,invalid_reason,relationship_id
0,19082112,19082112,19700101,20991231,None,Maps to
1,19082217,19082217,20090405,20991231,None,Maps to
2,19082289,19082289,19700101,20991231,None,Maps to
3,19082310,19082310,20080727,20991231,None,Maps to
4,19082311,19082311,19700101,20991231,None,Maps to
5,19082315,19082315,19700101,20991231,None,Maps to
6,19082320,19082320,19700101,20991231,None,Maps to
7,19082321,19082321,20090503,20991231,None,Maps to
8,19082331,19082331,20080727,20991231,None,Maps to
9,19082333,19082333,20080727,20991231,None,Maps to


In [33]:
subset_concept_sdf = concept_sdf.filter(F.col("vocabulary_id") == F.lit(ohdsi_vocabulary))
subset_concept_sdf.createOrReplaceTempView("subset_concept")

print(f"Total rows read: {subset_concept_sdf.count()}")

Total rows read: 99421


In [41]:
subset_concept_map_sdf_1 = subset_concept_sdf.alias("sc1").join(concept_relationship_sdf.alias("cr"), on=F.col("sc1.concept_id") == F.col("cr.concept_id_1"))\
.filter((F.col("cr.relationship_id") == F.lit("Maps to")) | (F.col("cr.relationship_id") == F.lit("Maps to Semantic Type")))
subset_concept_map_sdf_2 = subset_concept_map_sdf_1.join(concept_sdf.alias("sc2"), on=F.col("sc2.concept_id") == F.col("cr.concept_id_2"))
subset_concept_map_sdf_3 = subset_concept_map_sdf_2.select("cr.concept_id_1", "cr.relationship_id", "sc2.concept_id", "sc2.concept_code", "sc2.concept_name", "sc2.domain_id",	"sc2.concept_class_id", "sc2.standard_concept", "sc2.vocabulary_id")
subset_concept_map_sdf_4 = subset_concept_map_sdf_3.orderBy([F.col("cr.concept_id_1"), F.col("cr.relationship_id"), F.col("sc2.concept_id")])
print(f"Total rows read: {subset_concept_map_sdf_4.count()}")
subset_concept_map_sdf_4.limit(10).toPandas()


Total rows read: 128599


,concept_id_1,relationship_id,concept_id,concept_code,concept_name,domain_id,concept_class_id,standard_concept,vocabulary_id
0,8689,Maps to,46273463,10685111000119102,Upper respiratory tract infection caused by In...,Condition,Disorder,S,SNOMED
1,8690,Maps to,4194889,312836001,Enthesopathy of lower limb,Condition,Disorder,S,SNOMED
2,8691,Maps to,74104,106009009,Fetal condition affecting obstetrical care of ...,Condition,Disorder,S,SNOMED
3,8691,Maps to,80165,31805001,Fetal disproportion,Condition,Disorder,S,SNOMED
4,8692,Maps to,4113020,284607007,Finding relating to aggressive behavior,Observation,Clinical Finding,S,SNOMED
5,8693,Maps to,439237,52684005,Assault,Observation,Event,S,SNOMED
6,8694,Maps to,438046,269691005,Medical accident to patient during surgical an...,Observation,Event,S,SNOMED
7,9694,Maps to,4084229,24595009,Primary gout,Condition,Disorder,S,SNOMED
8,9695,Maps to,4116007,201663006,Gouty arthritis of the shoulder region,Condition,Disorder,S,SNOMED
9,9696,Maps to,4116007,201663006,Gouty arthritis of the shoulder region,Condition,Disorder,S,SNOMED


#### Convert Spark Dataframes into dictionaries

In [54]:
ohdsi_subset_concept_dict = key_nest_list_sdf(subset_concept_sdf[["concept_id",	"domain_id",	"concept_class_id",	"standard_concept", "concept_code"]], "concept_code")
print(f"Total entries: {len(ohdsi_subset_concept_dict)}")

Total entries: 99421


In [55]:
ohdsi_maps_to_dict = key_nest_list_sdf(subset_concept_map_sdf_4, "concept_id_1")
print(f"Total entries: {len(ohdsi_maps_to_dict)}")

Total entries: 98023


#### Nest dictionaries

In [56]:
nested_ohdsi_subset_dict = {}
for key in ohdsi_subset_concept_dict:
  entry = ohdsi_subset_concept_dict[key][0]
  concept_id = entry["concept_id"]
  if concept_id in ohdsi_maps_to_dict:
    entry["maps"] = ohdsi_maps_to_dict[concept_id]

  nested_ohdsi_subset_dict[key] = entry

print(f"Total entries: {len(nested_ohdsi_subset_dict)}")

Total entries: 99421


In [57]:
for row_dict in subset_vocab_dict:
  if row_dict["CODE"] in nested_ohdsi_subset_dict:
    row_dict["OHDSI"] = nested_ohdsi_subset_dict[row_dict["CODE"]]

In [58]:
subset_vocab_dict[-1]

{'AUI': 'A17811764',
 'LUI': 'L0808689',
 'CUI': 'C0481429',
 'ISPREF': 'N',
 'TTY': 'PT',
 'SAB': 'ICD10CM',
 'CODE': 'Z99.89',
 'STR': 'Dependence on other enabling machines and devices',
 'STY': [{'STN': 'A2.2', 'STY': 'Finding'}],
 'REL': [{'REL': 'PAR',
   'RELA': None,
   'AUI': 'A17850094',
   'TTY': 'HT',
   'STR': 'Dependence on other enabling machines and devices',
   'CODE': 'Z99.8'}],
 'hierarchies': {'None': [{'pos': 4,
    'col': 'A17850094',
    'AUI': 'A17850094',
    'CUI': 'C0481429',
    'TTY': 'HT',
    'CODE': 'Z99.8',
    'STR': 'Dependence on other enabling machines and devices'},
   {'pos': 3,
    'col': 'A17811761',
    'AUI': 'A17811761',
    'CUI': 'C0496751',
    'TTY': 'HT',
    'CODE': 'Z99',
    'STR': 'Dependence on enabling machines and devices, not elsewhere classified'},
   {'pos': 2,
    'col': 'A18913595',
    'AUI': 'A18913595',
    'CUI': 'C0478618',
    'TTY': 'HT',
    'CODE': 'Z77-Z99',
    'STR': 'Persons with potential health hazards related 

### Export results to JSONL

In [59]:
with open(UMLS_OUTPUT_DIRECTORY + f"umls_{sab}_subset_vocab_dict.jsonl", "w") as fw:
  for row_dict in subset_vocab_dict:
    json.dump(row_dict, fw)
    fw.write("\n")